# Playground Series S6E8 (Predicting Smartphone Addiction) / 「S6e8 Public Ensemble」解説付き写し

- **コンペ**: [Predicting Smartphone Addiction — Playground Series S6E8](https://www.kaggle.com/competitions/playground-series-s6e8)（Playground、3,196チーム、残り3日）
- **元notebook**: [S6e8 Public Ensemble](https://www.kaggle.com/code/kirill0212/s6e8-public-ensemble)
- **原著者**: CSTDY (kirill0212)
- **Public Score**: **0.97120**（Version 4、8 votes）
- **ライセンス**: Apache 2.0

## 手法の概要

自分でモデルを1つも学習しない、**純粋なスタッキング（stacking）専用notebook**。
公開されている他人のOOF予測ライブラリ（szymonkapiski の47モデル、adarsh1077、dariushafshar のgolem library、beicicc、
さらに donmarch14 の LightGBM/CatBoost notebook出力など）を数十本まとめて読み込み、
すべてを**ロジット空間に揃えて**から、強く正則化したロジスティック回帰（`C = 0.00599`）をメタモデルとして5-fold CVで学習する。
わずか200行足らずで LB 0.97120 に到達しており、「終盤のPlaygroundでは、**新しい特徴量よりも、良質な予測の集め方と混ぜ方**が効く」ことを端的に示している。

## 評価指標

- **タスク**: 二値分類。`addicted_label`（スマホ依存かどうか）を予測する。特徴量は年齢、1日のスクリーンタイム、SNS時間、ゲーム時間、勉強/仕事時間、睡眠時間、通知数、アプリ起動回数、週末スクリーンタイムの数値9個＋性別・ストレスレベル・学業/仕事への影響のカテゴリ3個。
- **指標**: **ROC AUC**。予測を降順に並べたとき「ランダムに選んだ陽性が、ランダムに選んだ陰性より高いスコアを得る確率」に等しい。
- **なぜこの指標か**: AUCは**予測値の順位だけ**を見る指標で、確率の絶対値やクラス比率に依存しません。閾値を決め打ちする必要もないため、Playgroundのような合成データの二値分類では標準的に使われます。裏返すと、**単調変換（ロジット化・順位化など）ではスコアが一切変わらない**ので、ブレンドの設計自由度が非常に高くなります。
- **このnotebookが指標に対してどう設計されているか**:
  1. 全ての予測を `prob_to_logit` で**ロジットに変換**してから混ぜている。AUCは単調変換不変なので変換自体は無害だが、確率のまま平均すると0や1付近の予測が潰れて情報が失われる。ロジット空間なら「かなり自信がある」という情報が線形結合に残る。
  2. メタモデルを `LogisticRegression(C=0.00599)` と**極端に強く正則化**している。`C` は正則化の逆数なので、`C≈0.006` は「係数をほぼゼロに引き寄せる」設定。数十本の高相関なOOFを入力にすると、重みが暴れて簡単にOOF上だけ過学習するため、**意図的に自由度を落として単純平均に近づけている**。
  3. `StratifiedKFold(5)` で層化分割し、fold内でメタモデルを学習してOOF AUCを測る。テスト予測はfold平均（bagging）。
  4. `fast_roc_auc_score` を自前実装（`np.trapezoid` でROC曲線下面積を台形則で積分）。数十本のモデルを個別評価するので、sklearnより軽い実装で回している。

> ⚠️ **断り書き**: これは学習目的の「解説付き写し」です。原著者のコード本体は変更しておらず、出力だけを削除した未実行状態です。各コードセルの直前に日本語解説Markdownセルを追加しています。


## Credits
Thanks to @szymonkapiski [`s6e8-oof-library-47-models`](https://www.kaggle.com/datasets/szymonkapiski/s6e8-oof-library-47-models), @beicicc, @adarsh1077 [`s6e8-adarsh-oof-library`](https://www.kaggle.com/datasets/adarsh1077/s6e8-adarsh-oof-library), @dariushafshar [`s6e8-golem-oof-library`](https://www.kaggle.com/datasets/dariushafshar/s6e8-golem-oof-library), @mohankrishnathalla, @ravi20076, @omidbaghchehsaraei, @donmarch14, @nawfeelrahman1124444, @zhenruiweng, @najiama, @tamerlanomralinov and others for sharing their models.

### 🔎 セル1の解説：設定・高速AUC実装・入力ファイルの一覧

**何をしているか**
ライブラリのインポートと定数定義に加え、2つの重要な準備をしています。
1. **`fast_roc_auc_score` の自前実装**: 予測値で降順ソート → 同値の切れ目を探す → 累積和でTP/FPを数える → 台形則（`np.trapezoid`）でROC曲線下の面積を求める、という素直な実装。
2. **`STACKING_FILES` の定義**: `(表示名, OOFファイルのパス, テスト予測ファイルのパス)` という3つ組のリスト。ここに数十本のモデルが列挙されます。パスは公開データセット（`/kaggle/input/datasets/...`）と、他人のnotebook出力（`/kaggle/input/notebooks/...`）の両方から来ています。

また `cuml`（GPU版scikit-learn）のインポートを `try/except` で囲み、**使えれば使う・無ければCPU版のまま**という書き方をしています。

**なぜそうするのか**
- **AUCを自前実装する理由**: 数十本のモデルそれぞれについてOOF AUCを計算するので、`sklearn.metrics.roc_auc_score` の入力検証などのオーバーヘッドが積み重なります。処理内容が単純なので自前化のコスパが良い。
- **他人のnotebook出力を直接入力にできる理由**: Kaggleでは「公開notebookの出力ファイル」をそのまま別notebookの入力として添付できます。これがPlaygroundの**協調的なスタッキング文化**の技術的基盤です。
- **`try/except` でGPUライブラリを囲む理由**: GPUが割り当てられていない環境でもnotebookが落ちないようにするため。可搬性を保つ定石。

**用語補足**
- **OOF（Out-Of-Fold）予測**: k-fold CVで、各サンプルが「検証側に回ったfold」のモデルから受け取った予測。学習に使われていないので、**訓練データ全体に対する疑似的なテスト予測**として扱えます。スタッキングの入力はこれでなければなりません（学習時の予測を使うとリークします）。
- **`EPS = 1e-7` / `LOGIT_CLIP = 30.0`**: 確率が厳密に0や1だとロジットが±∞に発散するので、微小量でクリップし、さらにロジット自体も±30で頭打ちにしています。

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 250)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
import gc
import os
import glob
from tqdm import tqdm
import polars as pl
try:
    from cuml.ensemble import RandomForestClassifier
    from cuml.linear_model import LogisticRegression
    import cuml
    cuml.set_global_output_type('numpy')
except:
    pass

ID = 'id'
TARGET = 'addicted_label'
NUMS = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
       'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time']
CATS = ['gender', 'stress_level', 'academic_work_impact']

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)
EPS         = 1e-7
LOGIT_CLIP  = 30.0
SEEDS = [42]

print(f'Seeds: {SEEDS}')

def fast_roc_auc_score(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    desc_score_indices = np.argsort(y_score)[::-1]
    y_true = y_true[desc_score_indices]
    y_score = y_score[desc_score_indices]
    distinct_value_indices = np.where(np.diff(y_score) != 0)[0]
    threshold_idxs = np.r_[distinct_value_indices, y_true.size - 1]
    tps = np.cumsum(y_true)[threshold_idxs]
    fps = 1 + threshold_idxs - tps
    tps = np.r_[0, tps]
    fps = np.r_[0, fps]
    fpr = fps / fps[-1]
    tpr = tps / tps[-1]
    return np.trapezoid(tpr, fpr)

def rreplace(text, old, new, count=1):
    parts = text.rsplit(old, count)
    return new.join(parts)

PATH_2 = '/kaggle/input/datasets/szymonkapiski/s6e8-oof-library-47-models/oof'
PATH_3 = '/kaggle/input/datasets/dariushafshar/s6e8-golem-oof-library'
PATH_4 = '/kaggle/input/datasets/adarsh1077/s6e8-adarsh-oof-library'
PATH_5 = '/kaggle/input/notebooks/omidbaghchehsaraei'
PATH_7 = '/kaggle/input/datasets/beicicc'
PATH_8 = '/kaggle/input/datasets/szymonkapiski/s6e8-50-weakest-oof-models'
PATH_9 = '/kaggle/input/datasets/boltuzamaki/s6e8-oof-prediction-library'
PATH_10 = '/kaggle/input/notebooks/redamountassir'
STACKING_FILES = [
    ('e-0', 
     f'/kaggle/input/notebooks/donmarch14/s6e8-lgbm/lgb_oof.npy', 
     f'/kaggle/input/notebooks/donmarch14/s6e8-lgbm/lgb_test.npy'),
    ('e-1', 
     f'/kaggle/input/notebooks/donmarch14/s6e8-catboost/oof_preds.csv', 
     f'/kaggle/input/notebooks/donmarch14/s6e8-catboost/test_preds.csv'),
    ('e-2', 
     f'/kaggle/input/notebooks/zhenruiweng/realmlp-for-predicting-smartphone-addiction/oof_preds.csv', 
     f'/kaggle/input/notebooks/zhenruiweng/realmlp-for-predicting-smartphone-addiction/submission.csv'),
    ('e-3', 
     f'/kaggle/input/notebooks/beicicc/s6e8-fold-safe-tabnet/tabnet_fold_safe_oof.csv', 
     f'/kaggle/input/notebooks/beicicc/s6e8-fold-safe-tabnet/tabnet_fold_safe_test.csv'),
    ('e-4', 
     f'/kaggle/input/notebooks/ravi20076/playgrounds6e8-public-baseline-v2/OOF_Preds_MLV2_1.parquet', 
     f'/kaggle/input/notebooks/ravi20076/playgrounds6e8-public-baseline-v2/Mdl_Preds_MLV2_1.parquet'),
    ('e-5', 
     f'/kaggle/input/notebooks/ravi20076/playgrounds6e8-public-baseline-v1/OOF_Preds_MLV1_1.parquet', 
     f'/kaggle/input/notebooks/ravi20076/playgrounds6e8-public-baseline-v1/Mdl_Preds_MLV1_1.parquet'),
    ('e-6', 
     f'/kaggle/input/datasets/mohankrishnathalla/s6e8-lgb-dart-oof/oof_lgb_v3.npy', 
     f'/kaggle/input/datasets/mohankrishnathalla/s6e8-lgb-dart-oof/test_lgb_v3.npy'),
    ('e-7', 
     f'/kaggle/input/datasets/mohankrishnathalla/s6e8-xgb-oof/oof_xgb_v3.npy', 
     f'/kaggle/input/datasets/mohankrishnathalla/s6e8-xgb-oof/test_xgb_v3.npy'),
    ('e-8', 
     f'/kaggle/input/datasets/mohankrishnathalla/s6e8-cat-mlp-oof/oof_cat_v3.npy', 
     f'/kaggle/input/datasets/mohankrishnathalla/s6e8-cat-mlp-oof/test_cat_v3.npy'),
    ('e-9', 
     f'/kaggle/input/notebooks/nawfeelrahman1124444/realmlp-0-97014/oof_preds.csv', 
     f'/kaggle/input/notebooks/nawfeelrahman1124444/realmlp-0-97014/submission.csv'),
]

paths_2 = glob.glob(f'{PATH_2}/*oof*', recursive=True)
for i, path in enumerate(paths_2):
    if path.find('knn') >= 0 or path.find('pub_evg') >= 0:
        continue
    STACKING_FILES.append((f's-{i}', path, rreplace(path, 'oof', 'test')))

paths_3 = glob.glob(f'{PATH_3}/*oof*', recursive=True)
for i, path in enumerate(paths_3):
    STACKING_FILES.append((f'd-{i}', path, rreplace(path, 'oof', 'test')))

paths_4 = glob.glob(f'{PATH_4}/*oof*', recursive=True)
for i, path in enumerate(paths_4):
    STACKING_FILES.append((f'a-{i}', path, rreplace(path, 'oof', 'test')))

for i, folder in enumerate(os.listdir(PATH_5)):
    STACKING_FILES.append((f'o-{i}', f'{PATH_5}/{folder}/oof.csv', f'{PATH_5}/{folder}/submission.csv'))

paths_7 = glob.glob(f'{PATH_7}/**/*oof*', recursive=True)
for i, path in enumerate(paths_7):
    STACKING_FILES.append((f'b-{i}', path, rreplace(path, 'oof', 'test')))

paths_10 = glob.glob(f'{PATH_10}/**/*oof*', recursive=True)
for i, path in enumerate(paths_10):
    STACKING_FILES.append((f'r-{i}', path, rreplace(path, 'oof', 'test')))

oof_arr = np.load(f'{PATH_8}/oof.npy')
test_arr = np.load(f'{PATH_8}/test.npy')
for i in range(50):
    np.save(f'oof_x-{i}.npy', oof_arr[:, i])
    np.save(f'test_x-{i}.npy', test_arr[:, i])
    STACKING_FILES.append((f'x-{i}', f'oof_x-{i}.npy', f'test_x-{i}.npy'))

oof_ = pd.read_parquet(f'{PATH_9}/oof_predictions.parquet')
test_ = pd.read_parquet(f'{PATH_9}/test_predictions.parquet')
for i, col in enumerate(oof_.columns):
    if col in ['id'] + ['deepfm_exact', 'ebm_exact']:
        continue
    np.save(f'oof_u-{i}.npy', oof_[col].values)
    np.save(f'test_u-{i}.npy', test_[col].values)
    STACKING_FILES.append((f'u-{i}', f'oof_u-{i}.npy', f'test_u-{i}.npy'))

print(f'Total models: {len(STACKING_FILES)}')

### 🔎 セル2の解説：予測の読み込みとロジット空間への正規化

**何をしているか**
コンペの `train.csv` / `test.csv` を読み、`STACKING_FILES` のパスを1本ずつ舐めて予測を行列に積み上げます。核となるのが2つの関数です。

- **`prob_to_logit(p)`**: `p` を `[1e-7, 1-1e-7]` にクリップしてから `log(p/(1-p))` を計算し、さらに `±30` でクリップ。
- **`load_preds_polars(path, expected_rows)`**: CSV / Parquet / `.npy` のどれでも読めるローダ。CSV/Parquetの場合は**「float型かつユニーク値が2より多い列」だけをスコア列とみなし、その最後の列**を採用します。

さらに賢いのが**入力形式の自動判定**です。読み込んだ値の絶対値の最大が1.1以下なら「確率」とみなしてそのままロジット化、
1.1を超えていたら「既にロジットである」とみなして一度シグモイドで確率に戻してから改めてロジット化する、という分岐になっています。
そして各ファイルの読み込みは `try/except` で囲まれ、失敗したら `SKIP` と表示して次へ進みます。

**なぜそうするのか**
- **形式のばらつきに耐える必要がある理由**: 数十人が別々に作ったファイルなので、列名も拡張子も、確率かロジットかも統一されていません。**厳密な仕様を要求する代わりに、緩く推測して吸収する**設計にしないと、そもそも集まりません。
- **`try/except` でスキップする理由**: 添付データセットの1つが更新されてパスが変わっただけで全体が落ちるのは致命的です。「落ちたモデルは黙って外し、残りで走る」ことで、パイプラインが**壊れにくく（robust）**なります。
- **ロジット空間で混ぜる理由**: 確率のまま平均すると、`0.999` と `0.9999` の差（ロジットでは 6.9 vs 9.2 で明確な差）がほぼ潰れます。AUCは順位で決まるので、この末端の解像度が効いてきます。

**注意点（この設計の弱み）**
「`|値| <= 1.1` なら確率」というヒューリスティックは、**ロジットが全て小さい範囲に収まっているモデル**（例: 予測が全部0付近）を誤って確率と判定します。実務でこの手のコードを書くときは、メタデータで形式を明示するのが安全です。

In [ ]:
PATH_6 = '/kaggle/input/competitions/playground-series-s6e8'
train = pd.read_csv(f'{PATH_6}/train.csv', index_col='id')
test = pd.read_csv(f'{PATH_6}/test.csv', index_col='id')

train_id = train.reset_index()['id']
test_id = test.reset_index()['id']

y = train[TARGET].values
N = len(y)
M = len(test)
print(f'Train: {N:,} | Test: {M:,}')

def prob_to_logit(p):
    p = np.clip(p, EPS, 1.0 - EPS).astype(np.float64)
    return np.clip(np.log(p / (1.0 - p)), -LOGIT_CLIP, LOGIT_CLIP).astype(np.float32)

def load_preds_polars(path, expected_rows=None):
    df = None
    if path.lower().endswith(".csv"):
        df = pl.read_csv(path)
    elif path.lower().endswith(".parquet"):
        df = pl.read_parquet(path)
    if df is not None:
        score_columns = [
            col for col in df.columns 
            if df[col].dtype.is_float() and df[col].n_unique() > 2
        ]
        if len(score_columns) > 1:
            print(f'{name} more than 1 score column in file\n')
        return df.select(score_columns[-1]).slice(0, expected_rows).to_numpy().reshape(-1, 1)

    arr = np.load(path)
    return arr.reshape(-1, 1)

loaded_oofs  = []
loaded_tests = []
display_names = []

print('\nLoading models:')
res_oof = pd.DataFrame(columns=['name', 'OOF AUC'])
i = 0
for name, oof_f, test_f in STACKING_FILES:
    try:
        o = load_preds_polars(oof_f, expected_rows=N)
        t = load_preds_polars(test_f, expected_rows=M)
        if np.abs(o).max() <= 1.1:
            o = prob_to_logit(o)
        else:
            o = 1/ (1+np.exp(-o))
            t = 1/ (1+np.exp(-t))
            o = prob_to_logit(o)
            print(f'{name} has values > 1.1')

        t = prob_to_logit(t)

        assert o.shape == (N, 1), f'OOF shape {o.shape} != {(N, 1)}'
        assert t.shape == (M, 1), f'Test shape {t.shape} != {(M, 1)}'

        np.save(f'oof_{name}.npy', o)
        np.save(f'test_{name}.npy', t)

        o = pd.DataFrame(o, columns=[name])
        t = pd.DataFrame(t, columns=[name])

        loaded_oofs.append(o)
        loaded_tests.append(t)
        display_names.append(name)

        oof_score = fast_roc_auc_score(y, np.array(o.iloc[:, 0]))
        res_oof.loc[i] = [name, round(oof_score, 5)]
        i += 1

        # print(f'  {name:20s} OOF={o.shape} TEST={t.shape} OOF AUC={oof_score:.5f}')

    except Exception as e:
        print(f'  SKIP {name} ({oof_f}, {test_f}): {e}')

n_models = len(loaded_oofs)
if n_models == 0:
    raise RuntimeError('No stacking files loaded. Update STACKING_FILES paths and rerun.')

print(f'\nModels: {n_models}')

X = pd.concat(loaded_oofs, axis=1).astype(np.float32)
X_test = pd.concat(loaded_tests, axis=1).astype(np.float32)

### 🔎 セル3の解説：各モデルのOOF AUCを降順表示

**何をしているか**
1行だけのセル。読み込んだ全モデルのOOF AUCを `res_oof` データフレームから降順に並べて表示します。

**なぜそうするのか**
スタッキングを始める前に、**素材の品質分布を目で見る**ためのステップです。ここで確認したいのは主に3点。
1. 最上位モデルのAUCはいくつか（メタモデルはこれを超えられて初めて価値がある）。
2. 明らかに壊れているモデル（AUC ≈ 0.5、あるいは0.5未満で符号が逆）が混ざっていないか。
3. 上位がどれくらいの幅に密集しているか。密集＝高相関＝**強い正則化が必要**というサイン。

このnotebookが `C = 0.00599` という極端な値を選んでいる根拠は、まさにこの「上位が密集した高相関の入力」という状況にあります。

In [ ]:
res_oof.sort_values(by=['OOF AUC'], ascending=False)

### 🔎 セル4の解説：ブレンド本体（5-fold ロジスティック回帰メタモデル）

**何をしているか**
`run_blend(preds_tmp, test_tmp, seed)` は、以下を5回（fold数）繰り返します。
1. `StratifiedKFold` で訓練/検証に分割。
2. 訓練側のOOF行列で `LogisticRegression(C=0.00599484, max_iter=1000)` を学習。
3. 検証側の予測を `p_val_x` の該当位置に埋め、テスト予測を `p_test_x` に `1/5` ずつ足し込む。
4. そのfoldの検証AUCと、**各入力モデルに付いた係数（`model.coef_`）をソートして表示**。

最後に全foldのOOFを結合し、fold平均AUCとテスト予測を返します。
コメントアウトされた `LogisticRegressionCV` が残っており、**`C` を探索した痕跡**が見えます。

**なぜそうするのか**
- **なぜメタモデルにロジスティック回帰か**: 入力が既に「各モデルのロジット」なので、必要なのは**良い重み付き和を見つけること**だけです。GBDTのような非線形メタモデルは自由度が高すぎ、数万行のOOFでは簡単に過学習します。線形＋強正則化が定石。
- **なぜ `C` をこんなに小さくするか**: `C` は正則化強度の逆数（L2ペナルティが `1/C` に比例）。`C=0.006` は「係数を強くゼロへ縮める」設定で、結果としてメタモデルは**加重平均に近い、なだらかな重み**を学びます。高相関な入力に対して自由な重みを許すと、片方に大きな正の重み・もう片方に大きな負の重みを付けて訓練データのノイズに合わせにいく（**多重共線性による係数の暴走**）ため、それを抑えています。
- **なぜ係数を毎fold印字するか**: fold間で係数が大きくブレる＝メタモデルが不安定というサインです。安定していれば、そのブレンドはprivate LBでも崩れにくいと期待できます。**「スコアだけでなく、重みの安定性を見る」**のは終盤の重要な判断材料です。

**用語補足**
- **StratifiedKFold（層化k分割）**: 各foldの陽性率を全体と揃えて分割する方法。不均衡データでfoldごとのスコアが暴れるのを防ぎます。
- **多重共線性（multicollinearity）**: 説明変数同士が強く相関していて、係数が一意に定まりにくくなる現象。

In [ ]:
def run_blend(preds_tmp, test_tmp, seed=SEED):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    p_val_x = np.zeros(len(preds_tmp))
    p_test_x = np.zeros(len(test_tmp))
    oof = []
    M = []
    P = []
    for fold, (tr_idx, val_idx) in tqdm(enumerate(skf.split(preds_tmp, y))):
        X_train = preds_tmp.iloc[tr_idx].reset_index(drop=True)
        X_val = preds_tmp.iloc[val_idx].reset_index(drop=True)
        y_train = y[tr_idx]
        y_val = y[val_idx]
        X_test = test_tmp.copy()
        val_ids = train_id.iloc[val_idx].reset_index(drop=True)
    
        X_train.columns = X_train.columns.astype(str)
        X_val.columns = X_val.columns.astype(str)
        X_test.columns = X_test.columns.astype(str)
        
        # model = LogisticRegressionCV(max_iter=1000, random_state=seed, Cs=10, scoring='roc_auc', cv=5)
        model = LogisticRegression(C=0.00599484, max_iter=1000)
    
        model.fit(
            X_train, y_train
        )
    
        p_val = model.predict_proba(X_val)[:, 1]
        p_test = model.predict_proba(X_test)[:, 1]
        p_val_x[val_idx] = p_val
        p_test_x += p_test / N_FOLDS
    
        val_metric = roc_auc_score(y_val, p_val)
        print(f'{val_metric:.6f}')
    
        p_val_2 = pd.DataFrame(p_val_x[val_idx], columns=[TARGET])
        p_val_2['id'] = val_ids
        oof.append(p_val_2)
        M.append(val_metric)
    
        p_test_2 = pd.DataFrame(p_test_x, columns=[TARGET])
        P.append(p_test_2)
        map_coef = pd.Series(model.coef_[0], index=X_train.columns)
        print(map_coef.sort_values())
        try:
            print(model.C_)
        except:
            pass
        del X_train, X_val, y_train, y_val
        gc.collect()
    
    oof_preds = pd.concat(oof, ignore_index=True).sort_values(by='id').reset_index(drop=True)
    print(f'Mean fold AUC: {np.mean(M):.6f}')
    p_test_final = np.mean(P, axis=0)
    test_preds = pd.DataFrame(p_test_final, columns=[TARGET])
    test_preds['id'] = test_id
    return oof_preds, test_preds

### 🔎 セル5の解説：シード平均と最終OOFスコア

**何をしているか**
`SEEDS = [42]`（現状は1個）のループで `run_blend` を呼び、複数シードの結果を平均します。
最後に **pooled OOF AUC**（全fold分のOOF予測をまとめた1本のAUC）を印字します。

**なぜそうするのか**
- **シード平均の意義**: CV分割の切り方によってメタモデルの重みは微妙に変わります。複数シードで平均すれば、**「たまたま良い分割を引いた」分の運**を薄められます。今は1シードですが、リスト形式にしてあるので `SEEDS = [42, 2024, 777]` にするだけで拡張できる設計になっています。
- **fold平均AUC と pooled OOF AUC の違い**: 前者は各foldのAUCの平均、後者は全サンプルを1つにまとめて計算したAUC。**後者の方がやや厳しく、実態に近い**値になります（fold間で予測のスケールがズレていると pooled 側が下がるため）。両方を見ると、キャリブレーションのズレに気づけます。

In [ ]:
oof_l = []
P = []
for seed in SEEDS:
    oof_preds_tmp, preds_final_tmp = run_blend(X, X_test, seed = seed)
    oof_l.append(oof_preds_tmp.set_index('id'))
    P.append(preds_final_tmp.set_index('id'))
    train_ids = oof_preds_tmp[['id']]
oof_preds = pd.DataFrame(np.mean(oof_l, axis=0))
oof_preds.columns = [TARGET]
oof_preds['id'] = train_id
print(f'Pooled OOF AUC: {roc_auc_score(y, oof_preds[TARGET]):.6f}')
test_preds = pd.DataFrame(np.mean(P, axis=0))
test_preds.columns = [TARGET]
test_preds['id'] = test_id

### 🔎 セル6の解説：submission.csv の書き出し

**何をしているか**
`id` とターゲット列 `addicted_label` の2列でCSVを書き出します。
タイムスタンプ付きファイル名を生成するコードがありますが、直後に `ts = ''` で無効化されています。

**なぜそうするのか**
Kaggleの提出は `/kaggle/working/submission.csv` という**決まったファイル名**を要求します。
実験中は `submission_20260829_07-15-00.csv` のように分けたいが、本番コミット時は固定名でなければならない。
`ts = ''` の1行で切り替えられるようにしてあるのは、実験と提出を1つのnotebookで兼ねるための小技です。

**最後に**: AUCは順位だけを見る指標なので、この最終出力の確率値そのものをキャリブレーションする必要はありません（したところでスコアは1ミリも変わりません）。

In [ ]:
import datetime

ts = datetime.datetime.now().strftime("_%Y%m%d_%H-%M-%S")
ts = ''

sub = pd.DataFrame({ID: test_id, TARGET: test_preds[TARGET]})
sub.to_csv(f'submission{ts}.csv', index=False)
sub.head()